# Chapter 32 — REINFORCE: Monte Carlo Policy Gradient

> **Note:** PyTorch is not available in the browser. This notebook uses a **pure-NumPy softmax policy** for illustration. For full PyTorch REINFORCE with neural network policies, run locally.

REINFORCE is the foundational **policy gradient** algorithm. Rather than learning a value function  
and deriving a policy from it, we **directly optimize the policy parameters** θ.

**Topics covered:**
1. Softmax policy (pure NumPy)
2. Log probability computation
3. Policy gradient theorem
4. REINFORCE update step
5. Why we need a baseline
6. Full REINFORCE loop on a simple environment

In [ ]:
import numpy as np
import random

---
## Part 1 — Softmax Policy

The softmax policy maps unnormalized **logits** (one per action) to a probability distribution:

$$\pi(a|s, \theta) = \frac{e^{\theta_{s,a}}}{\sum_{a'} e^{\theta_{s,a'}}}$$

In practice we subtract `max(logits)` for numerical stability before exponentiating.

In [ ]:
def softmax(logits):
    """Numerically stable softmax over a 1D array of logits."""
    e = np.exp(logits - np.max(logits))
    return e / e.sum()

# Example: 4-action logits
logits = np.array([1.0, 2.0, 0.5, -1.0])
probs = softmax(logits)

print("Logits: ", logits)
print("Probs:  ", np.round(probs, 4))
print(f"Sum:     {probs.sum():.6f}  (must be 1.0)")
print()
print("Key property: higher logit → higher probability")
print(f"Action 1 (logit=2.0) has highest prob: {probs[1]:.4f}")

---
## Part 2 — Log Probability

Policy gradients need **log π(a|s)**. We work in log space for numerical stability.

In [ ]:
def log_prob(logits, action):
    """Compute log pi(action | logits)."""
    probs = softmax(logits)
    return np.log(probs[action] + 1e-8)   # epsilon for numerical safety

logits = np.array([1.0, 2.0, 0.5, -1.0])
for a in range(4):
    lp = log_prob(logits, a)
    p  = softmax(logits)[a]
    print(f"action={a}: log π = {lp:.4f}  (π = {p:.4f})")

print()
print("Note: log probabilities are always ≤ 0.")
print("More likely actions have log-prob closer to 0.")

---
## Part 3 — Policy Gradient Theorem

The **policy gradient theorem** gives us:

$$\nabla_\theta J(\theta) \propto \mathbb{E}_{\tau \sim \pi} \left[ G_t \cdot \nabla_\theta \log \pi(A_t|S_t, \theta) \right]$$

For a **softmax policy**, the gradient of log π(a|logits) with respect to logits is:

$$\frac{\partial}{\partial \theta_i} \log \pi(a|\theta) = \mathbf{1}[i=a] - \pi(i|\theta)$$

This is the **one-hot vector for action a** minus the current probability vector.

In [ ]:
def log_prob_gradient(logits, action):
    """Gradient of log pi(action | logits) w.r.t. logits.
    
    d/d(logits_i) log pi(a) = 1(i==a) - pi(i)
    """
    probs = softmax(logits)
    grad = -probs.copy()   # start with -pi(i) for all i
    grad[action] += 1.0    # add 1 for the chosen action
    return grad

logits = np.array([0.0, 0.0, 0.0, 0.0])  # uniform policy
action = 2
grad = log_prob_gradient(logits, action)

print(f"Gradient of log π(a=2) w.r.t. logits:")
print(f"  grad = {grad}")
print(f"  Sum of grad (should be 0): {grad.sum():.6f}")
print()
print("Interpretation:")
print("  Positive entry at action 2: increase its logit → more likely")
print("  Negative entries elsewhere: those logits decrease relatively")

---
## Part 4 — REINFORCE Update Step

The REINFORCE parameter update (per timestep):

$$\theta \leftarrow \theta + \alpha \cdot G_t \cdot \nabla_\theta \log \pi(A_t|S_t, \theta)$$

This pushes the logits to increase the probability of actions that led to high return.

In [ ]:
def policy_gradient_step(logits, action, G, alpha):
    """REINFORCE update for a single (state, action, return) tuple.
    
    logits:  parameter vector (logits for softmax policy)
    action:  action taken
    G:       discounted return G_t from this timestep
    alpha:   learning rate
    
    Returns updated logits.
    """
    probs = softmax(logits)
    # Gradient of log pi(a) w.r.t. logits
    grad = np.zeros_like(logits)
    grad -= probs             # -pi(i) for all i
    grad[action] += 1.0       # +1 for the taken action
    return logits + alpha * G * grad

# Demonstrate: action 1 with positive return should become more likely
logits = np.array([0.0, 0.0, 0.0, 0.0])
print("Before update:")
print(f"  Probs: {softmax(logits)}")

# Update: took action 1, received positive return G=1.0
for step in range(5):
    logits = policy_gradient_step(logits, action=1, G=1.0, alpha=0.5)

print("After 5 updates (action=1, G=1.0):")
print(f"  Probs: {np.round(softmax(logits), 4)}")
print(f"  Action 1 probability increased from 0.25 to {softmax(logits)[1]:.4f}")

---
## Part 5 — Why Do We Need a Baseline?

Without a baseline, REINFORCE has **high variance**: all returns are positive, so  
even bad actions get reinforced if they happen to follow positive trajectories.

Adding a baseline **b(s)** doesn't introduce bias (it cancels in expectation) but reduces variance:

$$\theta \leftarrow \theta + \alpha (G_t - b(S_t)) \nabla_\theta \log \pi(A_t|S_t)$$

The most common baseline: **V(s)** (the value function estimate).  
This gives us: `(G_t - V(s))` which is the **advantage** A(s,a).

In [ ]:
# Demonstrate variance reduction with a baseline
import random
random.seed(0)
np.random.seed(0)

def simulate_returns(n=100, mean=5.0, std=10.0):
    """Simulate noisy returns from an episode."""
    return [random.gauss(mean, std) for _ in range(n)]

returns = simulate_returns(1000)
baseline = sum(returns) / len(returns)   # running mean as baseline
advantages = [G - baseline for G in returns]

import math
std_no_baseline = math.sqrt(sum(r**2 for r in returns) / len(returns))
std_with_baseline = math.sqrt(sum(a**2 for a in advantages) / len(advantages))

print("Variance Reduction with Baseline:")
print(f"  Mean return (baseline b): {baseline:.2f}")
print(f"  RMS of G (no baseline):   {std_no_baseline:.4f}")
print(f"  RMS of (G-b) w/ baseline: {std_with_baseline:.4f}")
print(f"  Variance reduction:        {1 - (std_with_baseline/std_no_baseline):.1%}")
print()
print("The baseline shifts returns to be centered near 0.")
print("Actions above average get positive updates; below average get negative.")
print("Crucially: E[b * grad log pi] = 0, so no bias is introduced.")

---
## Part 6 — Full REINFORCE Agent

Implement REINFORCE on a **3-armed bandit** (stateless environment).  
True rewards: arm 0 = 0.3, arm 1 = 0.7, arm 2 = 0.5.  
Goal: learn to prefer arm 1.

In [ ]:
# Stateless REINFORCE on a 3-armed bandit
# State is always 0 (single state), theta is a 3-vector of logits

TRUE_MEANS = [0.3, 0.7, 0.5]
n_actions = 3

def bandit_pull(arm):
    return random.gauss(TRUE_MEANS[arm], 0.5)

def reinforce_bandit(n_steps=5000, alpha=0.05, use_baseline=True, seed=42):
    random.seed(seed)
    np.random.seed(seed)

    theta = np.zeros(n_actions)
    baseline = 0.0
    beta = 0.99   # exponential moving average for baseline

    reward_history = []
    prob_arm1_history = []

    for step in range(n_steps):
        # Select action from softmax policy
        probs = softmax(theta)
        arm = np.random.choice(n_actions, p=probs)

        # Get reward
        r = bandit_pull(arm)
        reward_history.append(r)

        # Update baseline (running mean)
        if use_baseline:
            baseline = beta * baseline + (1 - beta) * r
            G = r - baseline
        else:
            G = r

        # REINFORCE update
        theta = policy_gradient_step(theta, arm, G, alpha)
        prob_arm1_history.append(softmax(theta)[1])

    return theta, reward_history, prob_arm1_history

theta_final, rewards, prob_arm1 = reinforce_bandit(use_baseline=True)

print("Final logits:", np.round(theta_final, 3))
print("Final policy:", np.round(softmax(theta_final), 4))
print(f"Arm 1 (best) probability: {softmax(theta_final)[1]:.4f}  (should dominate)")
print()
avg_reward = sum(rewards[-500:]) / 500
print(f"Mean reward (last 500 steps): {avg_reward:.4f}  (optimal ≈ {max(TRUE_MEANS):.2f})")

---
## Part 7 — Compare: With vs Without Baseline

In [ ]:
# Compare convergence speed with and without baseline
theta_b, rewards_b, p1_b     = reinforce_bandit(use_baseline=True,  seed=42)
theta_nb, rewards_nb, p1_nb  = reinforce_bandit(use_baseline=False, seed=42)

def window_avg(lst, window=200):
    return [sum(lst[max(0,i-window):i+1])/min(i+1,window) for i in range(len(lst))]

r_smooth_b  = window_avg(rewards_b)
r_smooth_nb = window_avg(rewards_nb)

# Print milestones
print("P(arm 1) over training (with vs without baseline):")
print(f"{'Step':>6}  {'With baseline':>15}  {'No baseline':>12}")
milestones = [0, 499, 999, 1999, 4999]
for i in milestones:
    if i < len(p1_b):
        print(f"{i+1:>6}  {p1_b[i]:>15.4f}  {p1_nb[i]:>12.4f}")

print()
print(f"Final mean reward (last 500, baseline):    {sum(rewards_b[-500:])/500:.4f}")
print(f"Final mean reward (last 500, no baseline): {sum(rewards_nb[-500:])/500:.4f}")
print()
print("Baseline typically leads to faster and more stable convergence.")

---
## Part 8 — REINFORCE on a Short Sequential Task

Extend REINFORCE to handle **sequential episodes** (not just bandits).  
Task: 3-state chain. Goal: reach state 2 from state 0.

In [ ]:
# 3-state chain: states 0,1,2. Actions: 0=left, 1=right.
# State 2 is terminal (reward=1). Other transitions: reward=0.
# One theta per state (each a 2-vector of logits).

def chain_step(s, a):
    if a == 1:  # right
        s_next = s + 1
    else:       # left
        s_next = max(0, s - 1)
    done = s_next >= 3
    r = 1.0 if done else 0.0
    return s_next, r, done

def collect_episode(thetas, gamma=0.99, max_steps=20):
    """Run one episode and return (states, actions, returns)."""
    s = 0
    trajectory = []  # (state, action, reward)
    for _ in range(max_steps):
        logits = thetas[s]
        probs = softmax(logits)
        a = np.random.choice(2, p=probs)
        s_next, r, done = chain_step(s, a)
        trajectory.append((s, a, r))
        if done:
            break
        s = s_next

    # Compute discounted returns (backward pass)
    returns = []
    G = 0.0
    for _, _, r in reversed(trajectory):
        G = r + gamma * G
        returns.insert(0, G)

    return trajectory, returns

random.seed(0)
np.random.seed(0)

# One theta (logits) per state, 2 actions each
thetas = [np.zeros(2) for _ in range(3)]
alpha_pg = 0.1
total_rewards = []

for ep in range(1000):
    trajectory, returns = collect_episode(thetas)
    ep_reward = sum(r for _, _, r in trajectory)
    total_rewards.append(ep_reward)

    # REINFORCE update for each step
    baseline = sum(returns) / len(returns)
    for (s, a, r), G in zip(trajectory, returns):
        advantage = G - baseline
        thetas[s] = policy_gradient_step(thetas[s], a, advantage, alpha_pg)

print("Learned policy (prob of going right = action 1):")
for s in range(3):
    p_right = softmax(thetas[s])[1]
    print(f"  State {s}: P(right) = {p_right:.4f}  (should be > 0.9)")

avg_final = sum(total_rewards[-100:]) / 100
print(f"\nAvg reward (last 100 eps): {avg_final:.4f}  (optimal = 1.0)")

---
## Part 9 — Exercise: Implement REINFORCE with Value Baseline

Add a learned **value function baseline** V(s) alongside the policy.  
The advantage becomes: A(s,a) = G_t − V(s).

V(s) is updated by MSE: `V(s) += beta * (G_t - V(s))`.

In [ ]:
random.seed(1)
np.random.seed(1)

thetas_v = [np.zeros(2) for _ in range(3)]
V_baseline = {0: 0.0, 1: 0.0, 2: 0.0}   # state value estimates
alpha_pi = 0.1
alpha_v  = 0.1   # learning rate for value baseline

rewards_v = []

for ep in range(1000):
    trajectory, returns = collect_episode(thetas_v)
    rewards_v.append(sum(r for _, _, r in trajectory))

    for (s, a, r), G in zip(trajectory, returns):
        # Compute advantage using value baseline
        advantage = G - V_baseline[s]

        # Update policy
        thetas_v[s] = policy_gradient_step(thetas_v[s], a, advantage, alpha_pi)

        # Update value baseline (gradient descent on MSE)
        V_baseline[s] += alpha_v * (G - V_baseline[s])

print("REINFORCE with Value Baseline:")
print("Learned policy:")
for s in range(3):
    p_right = softmax(thetas_v[s])[1]
    print(f"  State {s}: P(right) = {p_right:.4f}")

print("\nLearned value baseline:")
for s in range(3):
    print(f"  V({s}) = {V_baseline[s]:.4f}")

avg_v  = sum(rewards_v[-100:]) / 100
print(f"\nAvg reward (last 100): {avg_v:.4f}")

---
## Part 10 — Conceptual Questions

Answer these in the cell below.

In [ ]:
questions = {
    "Q1": "Why is REINFORCE called a Monte Carlo method?",
    "Q2": "What is the 'score function trick' (log-derivative trick)?",
    "Q3": "Why does REINFORCE have high variance? Name two ways to reduce it.",
    "Q4": "If G_t is always positive, what problem arises without a baseline?",
    "Q5": "What is Actor-Critic? How does it differ from REINFORCE with baseline?",
}

your_answers = {
    "Q1": "TODO",
    "Q2": "TODO",
    "Q3": "TODO",
    "Q4": "TODO",
    "Q5": "TODO",
}

for q, prompt in questions.items():
    print(f"{q}: {prompt}")
    print(f"    Your answer: {your_answers[q]}")
    print()

<details><summary>▶ Reference Answers</summary>

**Q1 — Monte Carlo:**  
REINFORCE uses the **full episode return** G_t (not bootstrapped). Like MC value methods, it waits until the end of the episode before making updates. This gives unbiased but high-variance gradient estimates.

**Q2 — Score function trick:**  
$$\nabla_\theta \mathbb{E}[f(x)] = \mathbb{E}[f(x) \nabla_\theta \log p_\theta(x)]$$  
This converts the gradient of an expectation into an expectation of a product, enabling Monte Carlo estimation without needing to differentiate through the sampling process.

**Q3 — High variance sources and reductions:**  
- *Sources:* (1) G_t depends on many random actions/transitions; (2) same (s,a) pair can yield very different returns depending on later randomness.  
- *Reductions:* (1) **Baseline subtraction** — subtract b(s) from G_t; (2) **Actor-Critic** — replace G_t with a 1-step TD target to reduce horizon of variance; (3) **GAE** (Generalized Advantage Estimation) — mix TD and MC.

**Q4 — Always positive returns:**  
If G_t > 0 always, every action gets a positive update — even bad ones. The policy gradient can still work (bad actions increase less than good ones), but convergence is much slower. A baseline centered around the average return fixes this.

**Q5 — Actor-Critic vs REINFORCE with baseline:**  
Both subtract a value baseline. The difference is **what the return estimate is**:  
- REINFORCE with baseline: uses full Monte Carlo G_t (waits for episode end)  
- Actor-Critic: uses **TD target** `r + γV(s')` as the critic's estimate, allowing **online** (per-step) updates. Lower variance but introduces bias.
</details>

---
## Summary

| Component | Role | Formula |
|-----------|------|----------|
| Softmax policy | Maps logits → probs | π(a\|θ) = exp(θₐ) / Σexp(θ) |
| Log probability | Objective term | log π(a\|θ) |
| Score gradient | ∇ of log prob | 1(i=a) − π(i) |
| REINFORCE update | Parameter step | θ ← θ + α·G_t·∇log π(a\|θ) |
| Baseline | Variance reduction | θ ← θ + α·(G_t−b)·∇log π(a\|θ) |
| Advantage | Return minus baseline | A(s,a) = G_t − V(s) |

**Running locally with PyTorch:**

```python
import torch
import torch.nn as nn
import torch.optim as optim

class PolicyNet(nn.Module):
    def __init__(self, n_obs, n_actions):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(n_obs, 64), nn.ReLU(), nn.Linear(64, n_actions))
    
    def forward(self, x):
        return torch.distributions.Categorical(logits=self.fc(x))

policy = PolicyNet(4, 2)   # e.g., CartPole
optimizer = optim.Adam(policy.parameters(), lr=1e-3)

# In the update loop:
# loss = -log_prob * G_t   (maximize expected return)
# loss.backward()
# optimizer.step()
```

**Next:** Chapter 33 — Actor-Critic Methods (A2C / PPO).